# 🛡️ XSS Attack Detection System — ML Training Pipeline
### By Harshit Kumar | GitHub: Harshit-Kumar-1710/The-Cross-Site-Scripting-Attack-Detection

**Pipeline Overview:**
1. ⚙️ CONFIG — All settings in one place
2. 📦 Install dependencies
3. 📚 Imports & setup
4. 📤 Upload datasets
5. 📥 Load all datasets & build hybrid corpus (Kaggle + Modern + SecLists + PayloadBox)
6. 🔧 Feature engineering (18 handcrafted security features + TF-IDF)
7. ✂️ Train/test split + SMOTE class balancing
8. 🤖 Train Random Forest with GridSearchCV auto-tuning
9. 📊 Evaluate model & plot results
10. 🎯 Auto-calculate severity thresholds
11. 📈 Feature importance analysis
12. 🧪 Live prediction test
13. 💾 Save all artifacts
14. 📦 Create ZIP & auto-download

---
> **Instructions:** Upload `XSS_dataset.csv` and `modern_xss_dataset_2022_2025.csv` when Cell 4 prompts you.
> Then click **Runtime → Run All** and wait ~10–15 minutes.

---
## ⚙️ CELL 1 — MASTER CONFIG
> 🔧 Change ONLY this cell to control the entire notebook

In [ ]:
# ============================================================
#  MASTER CONFIG — Edit here, nowhere else
# ============================================================

CONFIG = {
    # ---------- Dataset paths ----------
    "uploaded_csv_path" : "/content/XSS_dataset.csv",
    "modern_csv_path"   : "/content/modern_xss_dataset_2022_2025.csv",
    "csv_sentence_col"  : "Sentence",
    "csv_label_col"     : "Label",

    # ---------- External data sources ----------
    "fetch_seclists"   : True,
    "fetch_payloadbox" : True,
    "seclists_urls": [
        "https://raw.githubusercontent.com/danielmiessler/SecLists/master/Fuzzing/XSS/xss-payload-list.txt",
        "https://raw.githubusercontent.com/danielmiessler/SecLists/master/Fuzzing/XSS/XSS-Jhaddix.txt",
        "https://raw.githubusercontent.com/danielmiessler/SecLists/master/Fuzzing/XSS/XSS-RSNAKE.txt",
    ],
    "payloadbox_url": "https://raw.githubusercontent.com/payloadbox/xss-payload-list/master/Payload/xss-payload-list.txt",

    # ---------- TF-IDF ----------
    "tfidf_max_features" : 3000,
    "tfidf_analyzer"     : "char_wb",
    "tfidf_ngram_range"  : [2, 5],

    # ---------- Model ----------
    "test_size"    : 0.20,
    "random_state" : 42,
    "cv_folds"     : 5,
    "use_smote"    : True,
    "param_grid": {
        "n_estimators"     : [100, 200, 300],
        "max_depth"        : [None, 20, 40],
        "min_samples_split": [2, 5, 10]
    },

    # ---------- Severity thresholds (None = auto-calculate) ----------
    "severity_thresholds": None,

    # ---------- Output ----------
    "output_dir"            : "/content/xss_model_artifacts",
    "model_filename"        : "xss_model.pkl",
    "vectorizer_filename"   : "xss_vectorizer.pkl",
    "metadata_filename"     : "model_metadata.json",
    "feature_names_filename": "feature_names.json",
    "zip_filename"          : "xss_model_artifacts.zip",
}

print("✅ CONFIG loaded successfully")
print(f"   TF-IDF features  : {CONFIG['tfidf_max_features']}")
print(f"   Test size        : {CONFIG['test_size']}")
print(f"   SMOTE            : {CONFIG['use_smote']}")
print(f"   Fetch SecLists   : {CONFIG['fetch_seclists']}")
print(f"   Fetch PayloadBox : {CONFIG['fetch_payloadbox']}")

---
## 📦 CELL 2 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    "scikit-learn",
    "imbalanced-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "requests",
    "joblib",
    "tqdm",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"  ✅ {pkg}")

print("\n🎉 All dependencies installed!")

---
## 📚 CELL 3 — Imports & Setup

In [ ]:
import os, re, json, zipfile, warnings, requests, datetime, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score, f1_score
)
from scipy.sparse import hstack, csr_matrix
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("✅ All imports successful")
print(f"📁 Output directory: {CONFIG['output_dir']}")

---
## 📤 CELL 4 — Upload Your Datasets
> Upload **both** files when the picker appears:
> 1. `XSS_dataset.csv` — original Kaggle dataset
> 2. `modern_xss_dataset_2022_2025.csv` — modern synthetic dataset

In [ ]:
from google.colab import files

print("📤 Please upload BOTH files now:")
print("   1️⃣  XSS_dataset.csv")
print("   2️⃣  modern_xss_dataset_2022_2025.csv")
print()

uploaded = files.upload()

for fname, content in uploaded.items():
    if "modern" in fname.lower():
        save_path = CONFIG["modern_csv_path"]
    elif fname.lower().endswith(".csv"):
        save_path = CONFIG["uploaded_csv_path"]
    else:
        print(f"  ⚠️  Unrecognised file skipped: {fname}")
        continue
    with open(save_path, "wb") as fp:
        fp.write(content)
    print(f"  ✅ Saved: {save_path}")

# Quick validation
print()
df_check = pd.read_csv(CONFIG["uploaded_csv_path"], nrows=2)
print("🔍 Kaggle CSV columns :", df_check.columns.tolist())
print("🔍 Kaggle CSV shape   :", pd.read_csv(CONFIG["uploaded_csv_path"]).shape)

df_mod_check = pd.read_csv(CONFIG["modern_csv_path"], nrows=2)
print("🔍 Modern CSV columns :", df_mod_check.columns.tolist())
print("🔍 Modern CSV shape   :", pd.read_csv(CONFIG["modern_csv_path"]).shape)

print("\n✅ Upload complete!")

---
## 📥 CELL 5 — Load All Datasets & Build Hybrid Corpus

In [ ]:
all_frames   = []
source_stats = {}

# ------------------------------------------------------------------
# 5A. Kaggle CSV  (~13,686 rows)
# ------------------------------------------------------------------
print("📂 Loading Kaggle CSV...")
df_main = pd.read_csv(CONFIG["uploaded_csv_path"])
df_main = df_main[[CONFIG["csv_sentence_col"], CONFIG["csv_label_col"]]].copy()
df_main.columns = ["text", "label"]
df_main["source"] = "kaggle_csv"
df_main.dropna(subset=["text"], inplace=True)
df_main["text"] = df_main["text"].astype(str).str.strip()
df_main = df_main[df_main["text"].str.len() > 0]
all_frames.append(df_main)
source_stats["kaggle_csv"] = {"total": len(df_main), "xss": int(df_main["label"].sum())}
print(f"  ✅ Kaggle CSV       : {len(df_main):,} rows  (XSS: {df_main['label'].sum():,})")

# ------------------------------------------------------------------
# 5B. Modern Synthetic Dataset (2022-2025) — 22 attack categories
# ------------------------------------------------------------------
print("\n📂 Loading modern synthetic dataset (2022–2025)...")
try:
    df_modern = pd.read_csv(CONFIG["modern_csv_path"])
    df_modern = df_modern[["Sentence", "Label"]].copy()
    df_modern.columns = ["text", "label"]
    df_modern["source"] = "synthetic_2022_2025"
    df_modern.dropna(subset=["text"], inplace=True)
    df_modern["text"] = df_modern["text"].astype(str).str.strip()
    df_modern = df_modern[df_modern["text"].str.len() > 0]
    all_frames.append(df_modern)
    source_stats["synthetic_modern"] = {"total": len(df_modern), "xss": int(df_modern["label"].sum())}
    print(f"  ✅ Modern synthetic : {len(df_modern):,} rows  (XSS: {df_modern['label'].sum():,})")
    print(f"     Covers: URL-encoded, double-encoded, HTML entity, Unicode escape,")
    print(f"             base64, fromCharCode, mixed-case, string concat obfuscation,")
    print(f"             prototype pollution, PostMessage, DOM XSS, template injection,")
    print(f"             CSP bypass, Service Worker, Shadow DOM, Trusted Types, mXSS,")
    print(f"             framework XSS (Angular/React/Vue), CSS injection, fetch API abuse")
except FileNotFoundError:
    print("  ⚠️  modern_xss_dataset_2022_2025.csv not found — skipping")

# ------------------------------------------------------------------
# 5C. SecLists (auto-fetched from GitHub)
# ------------------------------------------------------------------
if CONFIG["fetch_seclists"]:
    print("\n🌐 Fetching SecLists XSS payloads...")
    seclists_payloads = []
    for url in CONFIG["seclists_urls"]:
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                lines = [
                    line.strip()
                    for line in r.text.splitlines()
                    if line.strip() and not line.startswith("#")
                ]
                seclists_payloads.extend(lines)
                print(f"  ✅ {url.split('/')[-1]}: {len(lines):,} payloads")
            else:
                print(f"  ⚠️  HTTP {r.status_code} — {url.split('/')[-1]}")
        except Exception as e:
            print(f"  ❌ Failed: {e}")
    if seclists_payloads:
        df_seclists = pd.DataFrame({"text": seclists_payloads, "label": 1, "source": "seclists"})
        all_frames.append(df_seclists)
        source_stats["seclists"] = {"total": len(df_seclists), "xss": len(df_seclists)}
        print(f"  📊 SecLists total: {len(df_seclists):,} XSS payloads")

# ------------------------------------------------------------------
# 5D. PayloadBox (auto-fetched from GitHub)
# ------------------------------------------------------------------
if CONFIG["fetch_payloadbox"]:
    print("\n🌐 Fetching PayloadBox XSS payloads...")
    try:
        r = requests.get(CONFIG["payloadbox_url"], timeout=30)
        if r.status_code == 200:
            lines = [
                line.strip()
                for line in r.text.splitlines()
                if line.strip() and not line.startswith("#")
            ]
            df_pb = pd.DataFrame({"text": lines, "label": 1, "source": "payloadbox"})
            all_frames.append(df_pb)
            source_stats["payloadbox"] = {"total": len(df_pb), "xss": len(df_pb)}
            print(f"  ✅ PayloadBox: {len(df_pb):,} XSS payloads")
        else:
            print(f"  ⚠️  HTTP {r.status_code} from PayloadBox")
    except Exception as e:
        print(f"  ❌ PayloadBox failed: {e}")

# ------------------------------------------------------------------
# 5E. Merge & Deduplicate
# ------------------------------------------------------------------
print("\n🔀 Merging all sources...")
df_all       = pd.concat(all_frames, ignore_index=True)
before_dedup = len(df_all)
df_all.drop_duplicates(subset=["text"], inplace=True)
after_dedup  = len(df_all)

print(f"  Before dedup : {before_dedup:,} rows")
print(f"  After dedup  : {after_dedup:,} rows")
print(f"  Duplicates   : {before_dedup - after_dedup:,} removed")

print(f"\n📊 Final Label Distribution:")
label_counts = df_all["label"].value_counts()
print(f"  XSS    : {label_counts.get(1, 0):,}")
print(f"  Benign : {label_counts.get(0, 0):,}")

print(f"\n📋 Sources Summary:")
for src, stats in source_stats.items():
    print(f"  {src:<25}: {stats['total']:>6,} total  |  {stats['xss']:>6,} XSS")

print(f"\n🎯 TOTAL UNIQUE SAMPLES: {after_dedup:,}")

---
## 🔧 CELL 6 — Feature Engineering (18 Handcrafted Security Features)

In [ ]:
def extract_security_features(text):
    """
    18 handcrafted security-aware features for XSS detection.
    Each feature targets a specific XSS attack vector.
    """
    t = str(text).lower()

    features = [
        # ---------- Script / Execution ----------
        int(bool(re.search(r'<script',             t))),   # F01: script tag
        int(bool(re.search(r'javascript:',         t))),   # F02: javascript protocol
        int(bool(re.search(r'on\w+=',              t))),   # F03: event handler
        int(bool(re.search(r'alert\s*\(',          t))),   # F04: alert()
        int(bool(re.search(r'eval\s*\(',           t))),   # F05: eval()
        int(bool(re.search(r'document\s*\.',       t))),   # F06: document object
        int(bool(re.search(r'window\s*\.',         t))),   # F07: window object
        int(bool(re.search(r'src\s*=\s*["\']?\s*http', t))), # F08: remote src

        # ---------- Encoding / Obfuscation ----------
        int(bool(re.search(r'%[0-9a-f]{2}',       t))),   # F09: URL encoding
        int(bool(re.search(r'&#x?[0-9a-f]+;',     t))),   # F10: HTML entity
        int(bool(re.search(r'\\u[0-9a-f]{4}',     t))),   # F11: Unicode escape
        int(bool(re.search(r'base64',              t))),   # F12: Base64
        int(bool(re.search(r'fromcharcode',        t))),   # F13: fromCharCode

        # ---------- Injection Vectors ----------
        int(bool(re.search(r'<iframe',             t))),   # F14: iframe
        int(bool(re.search(r'<img[^>]+src',        t))),   # F15: img src
        int(bool(re.search(r'data:',               t))),   # F16: data URI
        int(bool(re.search(r'vbscript:',           t))),   # F17: VBScript

        # ---------- Structural ----------
        len(re.findall(r'<[^>]+>', t)) / max(len(t), 1),  # F18: tag density ratio
    ]

    return features


FEATURE_NAMES = [
    'has_script_tag',
    'has_javascript_protocol',
    'has_event_handler',
    'has_alert',
    'has_eval',
    'has_document_access',
    'has_window_access',
    'has_remote_src',
    'has_url_encoding',
    'has_html_entity',
    'has_unicode_escape',
    'has_base64',
    'has_fromcharcode',
    'has_iframe',
    'has_img_src',
    'has_data_uri',
    'has_vbscript',
    'tag_density_ratio',
]

print("🔧 Extracting 18 security features from all samples...")
security_features = [
    extract_security_features(t)
    for t in tqdm(df_all['text'], desc="Extracting features")
]
X_security = np.array(security_features)

print(f"\n✅ Security features shape: {X_security.shape}")
print(f"\n📋 Feature activation rates (% of rows where feature = 1):")
for i, fname in enumerate(FEATURE_NAMES[:-1]):   # skip float F18
    rate = X_security[:, i].mean() * 100
    print(f"  {fname:<30}: {rate:5.1f}%")

---
## 🧠 CELL 7 — TF-IDF Vectorization

In [ ]:
print(f"🧠 Building TF-IDF vectorizer ({CONFIG['tfidf_max_features']} features, "
      f"analyzer='{CONFIG['tfidf_analyzer']}', "
      f"ngram={CONFIG['tfidf_ngram_range']})...")

vectorizer = TfidfVectorizer(
    max_features   = CONFIG['tfidf_max_features'],
    analyzer       = CONFIG['tfidf_analyzer'],
    ngram_range    = tuple(CONFIG['tfidf_ngram_range']),
    sublinear_tf   = True,
    min_df         = 2,
    strip_accents  = 'unicode',
)

X_tfidf = vectorizer.fit_transform(df_all['text'])
print(f"✅ TF-IDF matrix shape    : {X_tfidf.shape}")

# Combine TF-IDF + Security features into one sparse matrix
X_combined    = hstack([X_tfidf, csr_matrix(X_security)])
y             = df_all['label'].values
total_features = X_combined.shape[1]

print(f"✅ Combined feature matrix : {X_combined.shape}")
print(f"   TF-IDF features         : {X_tfidf.shape[1]}")
print(f"   Security features       : {X_security.shape[1]}")
print(f"   Total features          : {total_features}")

---
## ✂️ CELL 8 — Train/Test Split + SMOTE Balancing

In [ ]:
print("✂️  Splitting dataset (stratified)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y,
    test_size    = CONFIG['test_size'],
    random_state = CONFIG['random_state'],
    stratify     = y
)

print(f"  Train size  : {X_train.shape[0]:,} samples")
print(f"  Test size   : {X_test.shape[0]:,} samples")
print(f"  Train XSS   : {y_train.sum():,}  |  Benign: {(y_train == 0).sum():,}")

if CONFIG['use_smote']:
    print("\n⚖️  Applying SMOTE for class balancing...")
    smote = SMOTE(random_state=CONFIG['random_state'])
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print(f"  After SMOTE — XSS: {y_train.sum():,}  |  Benign: {(y_train == 0).sum():,}")
    print(f"  New train size    : {X_train.shape[0]:,}")
else:
    print("\n⚠️  SMOTE disabled (set use_smote=True in CONFIG to enable)")

print("\n✅ Data ready for training")

---
## 🤖 CELL 9 — Train Random Forest with GridSearchCV Auto-Tuning

In [ ]:
total_fits = (
    len(CONFIG['param_grid']['n_estimators'])
    * len(CONFIG['param_grid']['max_depth'])
    * len(CONFIG['param_grid']['min_samples_split'])
    * CONFIG['cv_folds']
)

print("🔍 Starting GridSearchCV hyperparameter tuning...")
print(f"   Param grid : {CONFIG['param_grid']}")
print(f"   CV folds   : {CONFIG['cv_folds']}")
print(f"   Total fits : {total_fits}  (may take 5–15 min on Colab free tier)")

rf_base = RandomForestClassifier(
    random_state  = CONFIG['random_state'],
    n_jobs        = -1,
    class_weight  = 'balanced'
)

cv_strategy = StratifiedKFold(
    n_splits     = CONFIG['cv_folds'],
    shuffle      = True,
    random_state = CONFIG['random_state']
)

grid_search = GridSearchCV(
    estimator  = rf_base,
    param_grid = CONFIG['param_grid'],
    cv         = cv_strategy,
    scoring    = 'f1',
    n_jobs     = -1,
    verbose    = 1,
    refit      = True
)

t0 = time.time()
grid_search.fit(X_train, y_train)
elapsed = time.time() - t0

best_model    = grid_search.best_estimator_
best_params   = grid_search.best_params_
best_cv_score = grid_search.best_score_

print(f"\n✅ Training complete in {elapsed / 60:.1f} minutes")
print(f"   Best params  : {best_params}")
print(f"   Best CV F1   : {best_cv_score:.4f}")

---
## 📊 CELL 10 — Evaluate Model

In [ ]:
print("📊 Evaluating model on held-out test set...\n")

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"  Accuracy  : {acc:.4f}  ({acc * 100:.2f}%)")
print(f"  F1 Score  : {f1:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Benign', 'XSS']))

# ----- Plots -----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Benign', 'XSS'],
    yticklabels=['Benign', 'XSS'],
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Probability Distribution
benign_proba = y_proba[y_test == 0]
xss_proba    = y_proba[y_test == 1]
axes[1].hist(benign_proba, bins=50, alpha=0.6, label='Benign', color='steelblue')
axes[1].hist(xss_proba,    bins=50, alpha=0.6, label='XSS',    color='crimson')
axes[1].set_title('XSS Probability Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('XSS Probability')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
eval_plot_path = os.path.join(CONFIG['output_dir'], 'model_evaluation.png')
plt.savefig(eval_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Evaluation chart saved: {eval_plot_path}")

---
## 🎯 CELL 11 — Auto-Calculate Severity Thresholds

In [ ]:
print("🎯 Calculating severity thresholds from real data distribution...")

xss_probas = y_proba[y_test == 1]

if CONFIG['severity_thresholds'] is None:
    thresholds = {
        'safe_max'   : float(np.percentile(y_proba,    45)),
        'low_max'    : float(np.percentile(xss_probas, 25)),
        'medium_max' : float(np.percentile(xss_probas, 50)),
        'high_max'   : float(np.percentile(xss_probas, 75)),
    }
    print("  ✅ Auto-calculated thresholds from data distribution")
else:
    thresholds = CONFIG['severity_thresholds']
    print("  ✅ Using manually specified thresholds")


def get_severity(prob):
    if prob <= thresholds['safe_max']:
        return 'SAFE'
    elif prob <= thresholds['low_max']:
        return 'LOW'
    elif prob <= thresholds['medium_max']:
        return 'MEDIUM'
    elif prob <= thresholds['high_max']:
        return 'HIGH'
    else:
        return 'CRITICAL'


SEVERITY_ICONS = {
    'SAFE'    : '🟢',
    'LOW'     : '🟡',
    'MEDIUM'  : '🟠',
    'HIGH'    : '🔴',
    'CRITICAL': '🚨',
}

print(f"\n  {'Level':<10} {'Probability Range':<30} Icon")
print(f"  {'-'*50}")
print(f"  {'SAFE':<10} 0.000 → {thresholds['safe_max']:.3f}                  🟢")
print(f"  {'LOW':<10} {thresholds['safe_max']:.3f} → {thresholds['low_max']:.3f}                  🟡")
print(f"  {'MEDIUM':<10} {thresholds['low_max']:.3f} → {thresholds['medium_max']:.3f}                  🟠")
print(f"  {'HIGH':<10} {thresholds['medium_max']:.3f} → {thresholds['high_max']:.3f}                  🔴")
print(f"  {'CRITICAL':<10} {thresholds['high_max']:.3f} → 1.000                  🚨")

severity_labels  = [get_severity(p) for p in y_proba]
severity_series  = pd.Series(severity_labels).value_counts()

print(f"\n  Severity distribution on test set:")
for level in ['SAFE', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL']:
    count = severity_series.get(level, 0)
    bar   = '█' * max(count // max(severity_series.max() // 20, 1), 1)
    print(f"  {SEVERITY_ICONS[level]} {level:<10}: {count:>5}  {bar}")

---
## 📈 CELL 12 — Feature Importance Analysis

In [ ]:
print("📈 Analyzing feature importances...")

importances = best_model.feature_importances_
n_tfidf     = CONFIG['tfidf_max_features']

# Security feature importances (last 18 columns)
sec_importances = importances[n_tfidf:]
sec_df = pd.DataFrame({
    'feature'   : FEATURE_NAMES,
    'importance': sec_importances
}).sort_values('importance', ascending=False)

# Top TF-IDF character n-gram importances (first n_tfidf columns)
tfidf_importances = importances[:n_tfidf]
top_idx   = np.argsort(tfidf_importances)[::-1][:15]
tfidf_vocab = vectorizer.get_feature_names_out()
tfidf_df  = pd.DataFrame({
    'ngram'     : [tfidf_vocab[i] for i in top_idx],
    'importance': tfidf_importances[top_idx]
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1 — Security feature importances
median_imp = sec_df['importance'].median()
colors1    = ['#e74c3c' if v > median_imp else '#3498db' for v in sec_df['importance']]
axes[0].barh(sec_df['feature'], sec_df['importance'], color=colors1)
axes[0].set_title('Security Feature Importances', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Importance Score')
axes[0].invert_yaxis()

# Plot 2 — Top TF-IDF character n-grams
axes[1].barh(tfidf_df['ngram'], tfidf_df['importance'], color='#2ecc71')
axes[1].set_title('Top 15 TF-IDF Character N-Grams', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance Score')
axes[1].invert_yaxis()

plt.tight_layout()
fi_plot_path = os.path.join(CONFIG['output_dir'], 'feature_importance.png')
plt.savefig(fi_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Feature importance chart saved: {fi_plot_path}")
print(f"\n🔍 Top 5 Security Features:")
for _, row in sec_df.head(5).iterrows():
    print(f"  🔴 {row['feature']:<30}: {row['importance']:.4f}")

---
## 🧪 CELL 13 — Live Prediction Test

In [ ]:
def predict_xss(payload_text):
    """Full prediction pipeline — reused in FastAPI backend."""
    text = str(payload_text)

    tfidf_feat = vectorizer.transform([text])
    sec_feat   = csr_matrix(np.array([extract_security_features(text)]))
    combined   = hstack([tfidf_feat, sec_feat])

    prob     = float(best_model.predict_proba(combined)[0][1])
    pred     = int(best_model.predict(combined)[0])
    severity = get_severity(prob)

    sec_vals  = extract_security_features(text)
    triggered = [FEATURE_NAMES[i] for i, v in enumerate(sec_vals[:-1]) if v == 1]

    return {
        'payload'            : text[:200],
        'is_xss'             : pred == 1,
        'xss_probability'    : round(prob, 4),
        'severity'           : severity,
        'triggered_features' : triggered,
    }


# ---- Test payload suite ----
test_cases = [
    # --- Benign ---
    ("Normal text",         "Hello, this is a normal sentence."),
    ("Safe HTML",           "<p class='intro'>Welcome to my website!</p>"),
    ("SQL query",           "SELECT * FROM users WHERE id = 1"),
    # --- Classic XSS ---
    ("Script tag",          "<script>alert('XSS')</script>"),
    ("Img onerror",         "<img src=x onerror=alert(document.cookie)>"),
    ("JS eval",             "javascript:eval(atob('YWxlcnQoMSk='))"),
    ("SVG onload",          "<svg onload=alert(1)>"),
    ("Iframe JS",           "<iframe src=javascript:alert(1)>"),
    # --- Obfuscated ---
    ("Mixed-case entity",   "<ScRiPt>alert&#40;1&#41;</ScRiPt>"),
    ("URL-encoded",         "%3Cscript%3Ealert(1)%3C%2Fscript%3E"),
    # --- Modern (2022-2025) ---
    ("Prototype pollution", "Object.prototype.__proto__={'x':1}; alert(document.cookie)"),
    ("Trusted Types bypass","trustedTypes.createPolicy('default',{createHTML:s=>s})"),
]

print("🧪 Live Prediction Tests")
print("=" * 72)
for label, payload in test_cases:
    result = predict_xss(payload)
    icon   = SEVERITY_ICONS[result['severity']]
    short  = payload[:55] + '...' if len(payload) > 55 else payload
    print(f"\n{icon} [{result['severity']:<8}]  Prob: {result['xss_probability']:.3f}  | {label}")
    print(f"   Payload  : {short}")
    if result['triggered_features']:
        print(f"   Triggers : {', '.join(result['triggered_features'][:4])}")

---
## 💾 CELL 14 — Save All Model Artifacts

In [ ]:
output_dir = CONFIG['output_dir']

# 1. Model
model_path = os.path.join(output_dir, CONFIG['model_filename'])
joblib.dump(best_model, model_path)
print(f"✅ Model saved      : {model_path}")

# 2. Vectorizer
vec_path = os.path.join(output_dir, CONFIG['vectorizer_filename'])
joblib.dump(vectorizer, vec_path)
print(f"✅ Vectorizer saved : {vec_path}")

# 3. Feature names
feat_path = os.path.join(output_dir, CONFIG['feature_names_filename'])
with open(feat_path, 'w') as fp:
    json.dump(
        {
            'security_features': FEATURE_NAMES,
            'tfidf_features'   : vectorizer.get_feature_names_out().tolist()
        },
        fp, indent=2
    )
print(f"✅ Feature names    : {feat_path}")

# 4. Metadata
metadata = {
    'model_type'         : 'RandomForestClassifier',
    'trained_at'         : datetime.datetime.now().isoformat(),
    'author'             : 'Harshit Kumar',
    'github'             : 'Harshit-Kumar-1710/The-Cross-Site-Scripting-Attack-Detection',
    'dataset_size'       : int(len(df_all)),
    'train_size'         : int(X_train.shape[0]),
    'test_size_samples'  : int(X_test.shape[0]),
    'total_features'     : int(total_features),
    'tfidf_features'     : int(CONFIG['tfidf_max_features']),
    'security_features'  : 18,
    'best_params'        : best_params,
    'best_cv_f1'         : round(float(best_cv_score), 4),
    'test_accuracy'      : round(float(acc), 4),
    'test_f1'            : round(float(f1), 4),
    'test_roc_auc'       : round(float(auc), 4),
    'severity_thresholds': thresholds,
    'severity_levels'    : ['SAFE', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'],
    'data_sources'       : list(source_stats.keys()),
    'smote_used'         : CONFIG['use_smote'],
    'tfidf_config'       : {
        'max_features': CONFIG['tfidf_max_features'],
        'analyzer'    : CONFIG['tfidf_analyzer'],
        'ngram_range' : CONFIG['tfidf_ngram_range'],
    }
}

meta_path = os.path.join(output_dir, CONFIG['metadata_filename'])
with open(meta_path, 'w') as fp:
    json.dump(metadata, fp, indent=2)
print(f"✅ Metadata saved   : {meta_path}")

print("\n" + "=" * 52)
print("📊  MODEL SUMMARY")
print("=" * 52)
print(f"  Dataset size : {len(df_all):,} samples")
print(f"  Accuracy     : {acc * 100:.2f}%")
print(f"  F1 Score     : {f1:.4f}")
print(f"  ROC-AUC      : {auc:.4f}")
print(f"  Best params  : {best_params}")
print("=" * 52)

---
## 📦 CELL 15 — Create ZIP & Auto-Download

In [ ]:
from google.colab import files

zip_path = os.path.join('/content', CONFIG['zip_filename'])

print("📦 Creating ZIP archive...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(output_dir)):
        fpath    = os.path.join(output_dir, fname)
        size_kb  = os.path.getsize(fpath) / 1024
        zf.write(fpath, arcname=fname)
        print(f"  + {fname:<42} ({size_kb:>7.1f} KB)")

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\n✅ ZIP created : {zip_path}  ({zip_size_mb:.1f} MB)")

print("\n📥 Starting download...")
files.download(zip_path)

print("\n🎉 DONE! Your model artifacts are downloading.")
print()
print("📋 Contents of xss_model_artifacts.zip:")
print("  xss_model.pkl          → Load with joblib.load() in FastAPI backend")
print("  xss_vectorizer.pkl     → Load alongside the model")
print("  model_metadata.json    → Model info, thresholds, performance metrics")
print("  feature_names.json     → All 3018 feature names")
print("  model_evaluation.png   → Confusion matrix + probability distribution")
print("  feature_importance.png → Security + TF-IDF feature importance charts")
print()
print("🚀 Next step: Build the FastAPI backend that loads xss_model.pkl")